# Week 5 — optimize, select, freeze and assess profile transfer

Week 4 supplies provisional modelling choices and robustness evidence. Week 5
may refine those choices on development countries before freezing the model.
Today's search for worry spacing $t$ and worry–harm allocation $\alpha$ belongs
to this Week 5 development stage, not to the original Week 4 experiments.

Read the main workflow in this order:

1. Search a bounded grid of worry coding and worry–harm allocation for each k.
2. Extend the search adaptively and confirm a fixed shortlist with fresh
   development resamples. Apply the unchanged structural and ARI criteria.
3. Review original-response meanings, individual-profile recovery and the
   additional distinctions contributed by larger k.
4. Select T5 using that evidence and check shallow-tree/single-feature explanations.
5. Freeze the exact development specification, then assess unchanged held-out
   assignment, original meanings, prevalence and country-specific limitations.

**Data boundary:** parameter search and model selection use development responses
only. In the intended workflow, freeze before examining held-out responses.
In our actual research history, B/E had already been evaluated on the held-out
countries before today's extension. Therefore T5's assessment is secondary,
not a newly untouched external validation. Moving notebook sections does not
change that chronology or erase the earlier decisions.

The original B/E regional checks, transfer, country distances and handoff remain
in **Appendix A** as historical analyses; they are not T5 regional validation.
Earlier stage-specific conclusions are retained to show how the decision evolved;
the current primary-model decision is in Sections 4–5. Saved outputs are preserved,
and all expensive recomputation switches remain off.

Historical artifact and implementation paths may still contain `week04`.
They are deliberately not renamed: doing so could break fingerprints and cached
results. Their location is provenance, not the revised weekly teaching sequence.


## Start here

1. Keep the course folder structure: `weekly_notebooks`, `23_wrp` and
   `course_artifacts` next to each other. These notebooks are not data downloads.
2. Use the packages in `requirements-objective1.txt`. The recorded versions are
   the ones used for the verification. Start a fresh Python kernel, then run
   all cells from top to bottom. You do not need to run another notebook first.
3. Leave the refit switches below at `False` to reproduce the saved analysis
   without launching the expensive searches. Cells still recalculate the checks
   and summaries shown in their code. A switch set to `True` writes new comparison
   outputs under `course_artifacts/objective1_simplified_checks`; it does not
   replace the frozen results.

**What is needed to reproduce this:** the 2023 source data, frozen country split,
prepared datasets, and saved model/experiment files referenced below. These are
deliberate scientific inputs. The notebooks do not reconstruct every historical
experiment from the raw CSV alone. The original full searches remain in the
linked archived notebooks. Only load the trusted local course `.joblib` files.

The order is: **Week 3 baseline → Week 4 K-means → Week 4 alternative methods →
Week 4 partition comparisons → Week 5 validation.** All saved inputs are unchanged.

### Packages

With Python 3.11, install once from the course folder:
`python -m pip install -r weekly_notebooks/requirements-objective1.txt`

In [ ]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from threadpoolctl import threadpool_limits


### Optional refits — off by default

In [ ]:
RUN_ONE_LOCAL_REFERENCE = False


In [ ]:
# Run from the course folder or weekly_notebooks. No absolute data paths.
ROOT = Path.cwd()
while not (ROOT / 'course_artifacts').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'course_artifacts').is_dir(), 'Open the course folder first.'
ART = ROOT / 'course_artifacts'
OUT = ART / 'objective1_simplified_checks'
OUT.mkdir(exist_ok=True)


In [ ]:
DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + d for d in DOMAINS]
HARM = ['harm_' + d for d in DOMAINS + ['work']]
SELF = [f + '_personal' for f in HARM]
KNOWN = [f + '_known_person' for f in HARM]
FEATURES = WORRY + SELF + KNOWN


In [ ]:
def country_weights(frame):
    # Each country has equal total mass; preserve relative survey weights within it.
    return (frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
            / frame.COUNTRY_ISO3.nunique()).to_numpy()


### Shared development geometry

Use the same complete-response feature transform as Week 4. The function takes
$t$ as `top`; tuning passes `top=1/q`. It estimates no parameters from held-out
responses. The declaration below does not fit a model.


In [ ]:
def profile_matrix(frame, top, alpha):
    values = frame[FEATURES].to_numpy(float)
    assert np.isfinite(values).all(), 'This analysis uses complete responses only.'
    worry = np.where(values[:, :6] == 2, top, values[:, :6]) / top
    mean_worry = worry.mean(axis=1, keepdims=True)
    # Preserve the original floating-point operation order for cache comparisons.
    return np.column_stack([
        (worry - mean_worry) * np.sqrt(alpha) / np.sqrt(6),
        mean_worry * np.sqrt(alpha) / np.sqrt(6),
        values[:, 6:] * np.sqrt((1 - alpha) / 2) / np.sqrt(7),
    ])


## 1. Development-only optimization of worry coding and worry–harm allocation

This analysis was approved on 16 October 2026, after Objective 1 had been frozen
and the held-out countries examined. It is not part of the original Week 4 search
and is not another untouched validation. B4/E8, their IDs, their held-out results
and the Week 6 input assignments remain unchanged.

### 1.1. Parameters, objective and computational budget

Use worry coding **0 / q / 1**. This is the previous normalized **0 / 1 / t**
coding with **t = 1/q**. Search q from .1 to .9 and alpha from .2 to .8.
The six centred worry coordinates, reduced mean-worry coordinate, fourteen
personal/known-person harm coordinates, complete-response population and
equal-country weighting remain exactly as above.

For each k = 3,...,10 separately, maximize

$$S(q,\alpha;k)=\min\{Q_{.10}(\mathrm{ARI}_{respondents}),
Q_{.10}(\mathrm{ARI}_{countries})\}.$$

Keep the previous filters: largest cluster ≤40%, k × largest share <3, and
median and lower-decile ARI ≥.80 in both tests. Do not compare raw inertia
across geometries or select a single k solely by its stability score.
ARI is unweighted and compares assignments of the full common development
population. Equal-country weights are used in fitting and profile descriptions.

The fixed protocol is:

1. A 6 × 5 parameter grid for each k: 240 reference settings.
2. Reject structurally ineligible references. Evaluate the others using ten
   original respondent samples and ten original country samples: 45 ARIs per test.
3. For each k, refine around its strongest coarse pilot using the midpoint to
   each adjacent grid value. This tests at most nine local points, not a new
   unrestricted search. Break exact score ties by minimum median ARI, then
   distance to q=.5/alpha=.5, then q and alpha.
4. Select two parameter settings per k from the completed pilots. Add original
   B4/E8 as benchmarks. Confirm all of these using the same **fresh 50 samples
   per test**, giving 1,225 ARIs per test per setting. Reuse only matching fit caches.
5. Report the best tested setting for every k, including failures to qualify.
   Inspect nearby settings, original-response profiles, burden trees and
   single-feature explanations. Numerical winners are **not automatically frozen**.

Fresh confirmation reduces dependence on the pilot resampling draws. It does
not create new independent countries, remove selection bias, or prove that an
estimated category spacing is a true psychological scale. The result is the
best tested setting within this bounded search, not a guaranteed global optimum.

In [ ]:
PARAMETER_SEARCH = ROOT / 'course_artifacts/week04_kmeans_parameter_optimization/8ecb97fd957f69f8'
parameter_protocol = json.loads((PARAMETER_SEARCH / 'protocol.json').read_text())
display(pd.Series(parameter_protocol['design'], name='predeclared_search'))
print('Saved progress:', json.loads((PARAMETER_SEARCH / 'progress.json').read_text()))

### 1.2. The model declaration is unchanged

Only the two representation parameters vary. The original feature transform
above accepts t, so pass t = 1/q. Original worry categories remain 0/1/2 in
the stored data and in the substantive descriptions.

In [ ]:
def fit_tuned_kmeans(frame, q, alpha, k, seed):
    matrix = profile_matrix(frame, top=1 / q, alpha=alpha)
    model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
    with threadpool_limits(limits=1):
        model.fit(matrix, sample_weight=country_weights(frame))
    return model

### 1.3. Reproduce or resume the search

Keep the switch off when reading results. The runner uses four workers, a
single-run lock and fingerprint-checked caches. It never loads held-out responses.
The implementation is available directly as readable Python:
[model and stability code](../.week04_parameter_search/search.py),
[coarse-to-fine runner](../.week04_parameter_search/run_search.py),
[original-response and explanatory checks](../.week04_parameter_search/report_results.py).

In [ ]:
RUN_PARAMETER_OPTIMIZATION = False
if RUN_PARAMETER_OPTIMIZATION:
    import subprocess
    import sys
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/run_search.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/report_results.py')], check=True)

### 1.4. Read the available evidence without refitting

Pilot scores are screening results, not final confirmation. The full-confirmation
table is the numerical comparison; the explanatory tables show what the resulting
partitions mean. A high-recall rule with poor precision does not define a cluster.
Treat all labels beginning T as provisional descriptions, not replacements for B/E.
Exact parameter values are in `selected_reference_specs.json` and the reference
caches. Rounded table values are for display, not exact re-assignment.

In [ ]:
for filename in ['coarse_results.csv', 'refinement_seeds.csv', 'shortlist_before_confirmation.csv',
                 'full_confirmation_results.csv', 'best_tested_setting_per_k.csv']:
    path = PARAMETER_SEARCH / filename
    if path.exists():
        table = pd.read_csv(path)
        columns = [c for c in ['k', 'q', 't', 'alpha', 'score', 'respondents_median', 'respondents_p10',
                   'countries_median', 'countries_p10', 'largest', 'dominance', 'stability_pass',
                   'selection_status', 'at_q_boundary', 'at_alpha_boundary', 'status'] if c in table]
        print(filename)
        display(table[columns].round(4))
for filename in ['frozen_baselines_on_confirmation_samples.csv', 'comparison_with_frozen_BE.csv',
                 'comparison_with_existing_Ward8.csv', 'profile_interpretations.csv',
                 'burden_tree_summary.csv', 'single_feature_summary.csv',
                 'parameter_neighbourhoods.csv', 'new_partition_comparisons.csv']:
    path = PARAMETER_SEARCH / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path).round(3))

In [ ]:
figure = PARAMETER_SEARCH / 'coarse_stability_surfaces.png'
if figure.exists():
    from IPython.display import Image
    display(Image(filename=str(figure)))

### 1.5. Interpretation and remaining decision

The search confirms the same numerically supported resolutions found before: k = 3, 4, 6, 7 and 8. The confirmed settings for k = 5, 9 and 10 fail the unchanged stability requirements. This means that none of the shortlisted settings qualifies; it does not prove that no qualifying setting exists anywhere in the continuous parameter space.

The best tested four-cluster setting uses q = .7 (equivalently t = 10/7) and alpha = .5. Its minimum lower-decile ARI is .9617, compared with .8559 for B4 on the same fresh samples. Both respondent and country tests improve. This is a real improvement in the measured resampling stability, not evidence by itself that the scientific profiles are better. The new labels have ARI .593 with B4 and 83.1% matched equal-country-weighted membership agreement. They are therefore a changed partition, not simply a numerically cleaner copy of B4.

The four new profiles contain: broad known-person harm exposure (17.3%); a low-worry group especially marked by no crime worry (27.6%); low food/water worry but more crime worry (26.8%); and stronger food/water worry (28.2%). These are descriptive interpretations, not new frozen names.

There is a substantive warning. For tuned cluster T4-C1, the rule "crime worry = not worried" is selected in all five country-grouped folds. Its mean balanced accuracy is 96.0%, precision 82.5% and recall 99.96%. The corresponding original B4 rule was much less reconstructive: 82.2% balanced accuracy, 76.7% precision and 76.2% recall. T4 is not wholly a burden-score partition—the depth-2 burden tree has only 57.5% balanced accuracy—but one cluster is now much closer to a single-feature definition. This resembles the type of concern raised in Week 3, though it is not identical to the old water-harm finding. No new automatic rejection threshold is introduced after seeing these results.

The highest-stability three-cluster setting also carries a strong single-feature warning. For T3-C2, "not worried about crime" gives 97.3% balanced accuracy, 92.5% precision and 98.3% recall. Its high stability should not alone make it the preferred multidomain profile solution.

For eight clusters, the best tested setting is q = 4/15 (t = 3.75), alpha = .8. Its minimum lower-decile ARI is .8235, against .8170 for E8 on the same fresh samples. The gain is small, and the selected alpha lies at the upper search boundary. We cannot claim that .8 is a uniquely identified interior optimum. The new labels have ARI .791 with E8 and 77.6% matched weighted membership agreement. The profiles continue to distinguish low worry and different combinations of very high domain-specific worry; they are not a simple subdivision of tuned T4.

An important additional observation is stronger agreement with the previously retained Ward eight-cluster partition: ARI .901 and 88.7% matched weighted agreement, compared with approximately .820 and 82.3% for original E8. Ward was fitted at its original t = 3, alpha = .75, not at the newly tuned parameters. This is agreement across existing partitions with different representations, not a new matched-geometry method test and not independent external validation.

The tuned eight-cluster labels remain poorly reconstructed by the two burden scores (depth-2 balanced accuracy .304). Several single-feature rules have recall near one but low precision, between roughly .21 and .57. This does not reduce the eight-way partition to any one feature. It remains important to distinguish a feature shared by all cluster members from a feature that uniquely identifies them.

The seven-cluster setting shows why the pilot cannot be the final evidence. Its pilot score was .9386; fresh full confirmation gives .8646. The original seven-cluster setting scores .8580 on those same fresh samples. Thus the confirmed gain is small, despite the much larger pilot difference. Pairwise ARIs share fitted partitions and should not be treated as independent observations for a simple significance test.

There is no single preferred worry coding across resolutions: the coarser solutions favour a higher position for "somewhat worried", while the richer solutions retain a stronger distinction between "somewhat" and "very". These parameters optimize this stability objective conditional on k, the geometry, the filters and this population. They do not estimate a uniquely correct psychological response scale.

Decision boundary: retain B4/E8 as the frozen course and downstream reference models for now. The tuned solutions are documented exploratory alternatives. T4 offers a large stability gain but a stronger single-feature concern; T8 offers a small stability gain and much stronger agreement with the existing Ward partition. Neither should replace B/E automatically. Such a decision must consider profile meaning, the single-feature evidence and the lack of a new untouched country-validation set. No held-out responses, frozen assignments, maps, Week 6 results or manuscript text were changed by this search.

## 2. Bounded adaptive optimization of worry coding and allocation

**Objective.** Search between the previous grid points without repeatedly
expanding the grid. This follow-up was approved after reviewing Section 1.
It is exploratory development analysis, not another independent validation.
The original search, frozen B4/E8 assignments and Week 6 results are preserved.

### 2.1. What changes, and what stays fixed

Optimize $q=1/t$ and $\alpha$, independently for every $k=3,\ldots,10$.
The worry coding is $(0,q,1)$ with $q\in[0.1,0.9]$; the worry allocation is
$\alpha\in[0.2,0.8]$. These are declared modelling bounds, not estimated
psychological limits. An optimum on a boundary does not establish an interior
optimum and does not trigger an automatic expansion of the range.

Keep the same 109,488 complete-response development respondents, 113 countries,
separate self/known-person harm features, equal-country weights and K-means
settings. No imputation, z-standardization, PCA or held-out responses are used.

For respondent $i$, let $m_i$ be the mean of the six normalized worry responses.
The worry coordinates remain
$\sqrt{\alpha/6}\,(w_{i1}-m_i,\ldots,w_{i6}-m_i,m_i)$;
the fourteen harm coordinates remain
$\sqrt{(1-\alpha)/14}\,h_{i\ell}$.
There are six underlying worry measurements, not seven independent features.
The transformation deliberately reduces the squared-distance contribution of
a uniform change in all six worries by a factor of six relative to uncentered
worry. The denominators set maximum block-distance budgets, not necessarily
the observed worry/harm contributions.

### 2.2. Search, shortlist and fresh confirmation

1. Import the 301 previous reference settings and their original ten-resample
   pilot evidence. Earlier full-confirmation scores are not the search objective.
2. Use Optuna 4.5.0 multivariate TPE to propose **20 additional parameter pairs
   per k**, at most 160 new pairs. Each k has its own fixed random seed.
   Proposals depend on earlier results, rather than a new fixed grid. The grid
   and midpoint-refinement fields inherited in the saved protocol describe the
   previous evidence only; the active extension is the adaptive search.
3. Screen the reference partition structurally before resampling. Require
   largest equal-country-weighted cluster share $\leq0.40$, dominance
   $D=k\times\text{largest share}<3$, and the requested number of clusters.
4. Evaluate eligible proposals using the **same ten respondent and ten country
   samples as the earlier pilots**: 45 pairwise ARIs per test. These are search
   scores, not final evidence. Maximize the smaller of the two lower-decile ARIs.
5. Select two completed, structurally eligible settings per k. Break exact ties
   by minimum median ARI, distance to $(q,\alpha)=(0.5,0.5)$, then q and alpha.
   Save and hash this shortlist before confirmation. Add unchanged B4 and E8.
6. Confirm the fixed shortlist with **50 fresh samples per test**, 1,225 ARIs
   per test. The new samples are checked against every previous original and
   confirmation sample. Different fit seeds are also used.
7. Apply the unchanged rule: **median and lower-decile ARI $\geq0.80$ in both
   tests**, together with all structural filters. Report rejected settings too,
   but label them clearly. No model is frozen automatically.

The respondent test samples 80% within every development country. The country
test selects 90 of 113 countries, stratified by region and income. ARI compares
the full common development population and is unweighted; fitting and substantive
profile summaries use equal-country weights. Singleton country strata are always
represented, a limitation inherited from the original protocol.

Fresh samples reduce dependence on the pilot draws. They do not provide new
countries, eliminate all selection bias, or establish a unique true response
scale. The prior search and prior held-out findings were already known. This
bounded search cannot guarantee a global optimum.

**Preflight correction.** An initial proposed confirmation seed range overlapped
the earlier confirmation range. The all-pairs check stopped it before any new
fits. That attempt is preserved with a failure record. The active protocol uses
a disjoint range; this was not a clustering failure or a change in the criteria.

In [ ]:
ADAPTIVE_SEARCH = ROOT / 'course_artifacts/week04_kmeans_adaptive_optimization/54d309fe37735497'
adaptive_protocol = json.loads((ADAPTIVE_SEARCH / 'protocol.json').read_text())
display(pd.Series(adaptive_protocol['design'], name='adaptive_protocol'))
status_file = ADAPTIVE_SEARCH / 'completion.json'
print(json.loads(status_file.read_text()) if status_file.exists() else 'Search or confirmation still running; no final conclusions yet.')

### 2.3. Run or resume deliberately

The switch is off for ordinary reading. The runner uses four workers, resumable
fit caches and saved optimizer states. It shares a lock with the earlier search,
so duplicate runs are rejected. It checks the input data, model code, previous
results and runtime before resuming. Optuna is an additional search dependency;
it does not replace the ordinary scikit-learn KMeans model.

Implementation: [adaptive runner](../.week04_parameter_search/adaptive_search.py),
[unchanged model and stability functions](../.week04_parameter_search/search.py),
[verification](../.week04_parameter_search/test_adaptive.py).

In [ ]:
RUN_ADAPTIVE_OPTIMIZATION = False
if RUN_ADAPTIVE_OPTIMIZATION:
    import subprocess
    import sys
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/adaptive_search.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/report_adaptive.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search/test_adaptive.py')], check=True)

### 2.4. Numerical eligibility and substantive evidence

The first table contains only the best confirmed setting per k that passes
every numerical filter, plus B/E as benchmarks on the same fresh samples.
Their stability-pass flags are shown explicitly. It is not a list
of automatically adopted models. Subsequent tables retain the full evidence,
including failed qualifications, profile interpretations, burden trees and
single-feature reconstructions. Exact parameters remain in the saved specifications.

In [ ]:
best_path = ADAPTIVE_SEARCH / 'best_tested_setting_per_k.csv'
if best_path.exists():
    adaptive_best = pd.read_csv(best_path)
    eligible = adaptive_best.loc[adaptive_best.stability_pass].copy()
    eligible['model'] = 'Adaptive k' + eligible.k.astype(str)
    baseline = pd.read_csv(ADAPTIVE_SEARCH / 'frozen_baselines_on_confirmation_samples.csv')
    baseline['model'] = baseline.baseline
    comparison = pd.concat([eligible, baseline], ignore_index=True).sort_values(['k', 'model'])
    display(comparison[['model', 'k', 'q', 't', 'alpha', 'respondents_median', 'respondents_p10',
                        'countries_median', 'countries_p10', 'score', 'largest', 'dominance', 'stability_pass']].round(4))
    for filename in ['best_tested_setting_per_k.csv', 'full_confirmation_results.csv',
                     'comparison_with_frozen_BE.csv', 'comparison_with_existing_Ward8.csv',
                     'comparison_with_previous_tuned_models.csv', 'profile_interpretations.csv',
                     'burden_tree_summary_with_baselines.csv', 'single_feature_summary_with_baselines.csv',
                     'parameter_neighbourhoods.csv', 'new_partition_comparisons.csv']:
        path = ADAPTIVE_SEARCH / filename
        if path.exists():
            print(filename)
            display(pd.read_csv(path).round(4))
else:
    print('No final confirmation summary saved yet.')

In [ ]:
figure = ADAPTIVE_SEARCH / 'adaptive_search_surfaces.png'
if figure.exists():
    from IPython.display import Image
    display(Image(filename=str(figure)))

### 2.5. Interpretation and adoption decision

The adaptive search imported 301 earlier settings and evaluated exactly 20 additional pairs for each k, 160 in total. The two strongest eligible pilots per k were fixed before fresh confirmation; B4/E8 were included unchanged. No new grid was run and no search bounds, structural filters or ARI thresholds were changed.

The resolutions whose best confirmed setting passes every numerical filter are: 3, 4, 5, 6, 7, 8, 9. Other resolutions are reported as best tested but not retained. This bounded search cannot establish that no acceptable setting exists elsewhere.

For k=3, the strongest confirmed setting has q=0.622606, t=1.606152, alpha=0.739962; lower-decile respondent ARI=0.9856, country ARI=0.9811, and largest cluster=35.8%. It passes all numerical filters. 

For k=4, the strongest confirmed setting has q=0.592946, t=1.686495, alpha=0.430382; lower-decile respondent ARI=0.9815, country ARI=0.9703, and largest cluster=37.3%. It passes all numerical filters. 

For k=5, the strongest confirmed setting has q=0.416701, t=2.399803, alpha=0.583786; lower-decile respondent ARI=0.9465, country ARI=0.9131, and largest cluster=36.3%. It passes all numerical filters. 

For k=6, the strongest confirmed setting has q=0.315149, t=3.173098, alpha=0.664834; lower-decile respondent ARI=0.8486, country ARI=0.8424, and largest cluster=36.2%. It passes all numerical filters. 

For k=7, the strongest confirmed setting has q=0.333333, t=3.000000, alpha=0.625000; lower-decile respondent ARI=0.8799, country ARI=0.9335, and largest cluster=34.0%. It passes all numerical filters. 

For k=8, the strongest confirmed setting has q=0.369154, t=2.708900, alpha=0.626338; lower-decile respondent ARI=0.8647, country ARI=0.8599, and largest cluster=33.1%. It passes all numerical filters. 

For k=9, the strongest confirmed setting has q=0.380574, t=2.627611, alpha=0.669347; lower-decile respondent ARI=0.8201, country ARI=0.8129, and largest cluster=32.3%. It passes all numerical filters. 

For k=10, the strongest confirmed setting has q=0.634330, t=1.576466, alpha=0.798567; lower-decile respondent ARI=0.6686, country ARI=0.6814, and largest cluster=21.4%. It fails the unchanged stability requirements. 

On the same fresh samples, original B4 has score 0.9145, compared with 0.9703 for the best confirmed k=4 setting. The reference partitions have ARI 0.3318 and matched equal-country-weighted agreement 55.2%. A stability increase alone does not establish that the new profiles are scientifically preferable.

On the same fresh samples, original E8 has score 0.8162, compared with 0.8599 for the best confirmed k=8 setting. The reference partitions have ARI 0.8074 and matched equal-country-weighted agreement 82.5%. A stability increase alone does not establish that the new profiles are scientifically preferable.

For T3, the strongest single-feature reconstruction is cluster 2, usually using worry_crime (5/5 folds): balanced accuracy 97.7%, precision 92.8%, recall 98.8%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T4, the strongest single-feature reconstruction is cluster 3, usually using harm_traffic_known_person (5/5 folds): balanced accuracy 94.2%, precision 67.4%, recall 100.0%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T5, the strongest single-feature reconstruction is cluster 4, usually using worry_water (5/5 folds): balanced accuracy 88.3%, precision 67.4%, recall 84.3%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T6, the strongest single-feature reconstruction is cluster 4, usually using worry_mental_health (5/5 folds): balanced accuracy 94.0%, precision 59.1%, recall 99.5%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T7, the strongest single-feature reconstruction is cluster 3, usually using worry_mental_health (5/5 folds): balanced accuracy 93.4%, precision 52.4%, recall 100.0%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T8, the strongest single-feature reconstruction is cluster 2, usually using worry_mental_health (5/5 folds): balanced accuracy 92.7%, precision 46.5%, recall 99.9%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

For T9, the strongest single-feature reconstruction is cluster 8, usually using worry_food (5/5 folds): balanced accuracy 92.5%, precision 36.5%, recall 99.8%. Read precision and recall together. This is a diagnostic of existing cluster membership, not a new clustering or an automatic rejection rule.

T7 agrees with the existing Ward8 partition at ARI 0.6782. Ward was not refitted at the new parameters: this is agreement between existing partitions with possibly different geometries, not a new matched-geometry algorithm validation.

T8 agrees with the existing Ward8 partition at ARI 0.7399. Ward was not refitted at the new parameters: this is agreement between existing partitions with possibly different geometries, not a new matched-geometry algorithm validation.

Scientific reading of the coarser solutions. T3 is extremely stable, but one of its three groups is almost identified by not worrying about crime (97.7% balanced accuracy and 92.8% precision). This is a strong feature-dependence concern, not evidence that the numerical tests failed. T4 is not a small adjustment of B4: their ARI is only .332 and matched weighted agreement is 55.2%. Its four groups distinguish low food/water worry with more traffic/weather worry (28.6%), broad self/known-person harm exposure (14.7%), generally lower worry and harm (37.3%), and known-person traffic harm (19.4%). All members of the last group report known-person traffic harm, but that response also occurs outside the group: rule precision is only 67.4%. It is therefore a necessary marker in this fitted group, not a unique single-feature definition. The high stability supports this alternative partition; it does not make it the same scientific result as B4.

Newly supported intermediate resolution. T5 passes both full tests at score .9131. Its provisional groups are lower worry (36.3%), broad known-person harm exposure (13.8%), stronger crime/weather worry (18.5%), stronger mental-health/traffic/weather worry (15.8%), and stronger water/food worry (15.5%). These descriptions summarize contrasts in the original categories, not final profile names. Its strongest single-feature rule has 88.3% balanced accuracy, 67.4% precision and 84.3% recall; the depth-2 burden tree has 46.0% balanced accuracy. It is a useful intermediate-resolution candidate to compare substantively with B/E, not an automatically adopted replacement.

Stability and method agreement do not identify the same winner. The selected adaptive T8 has score .8599, above E8 at .8162 on the same samples, while retaining a low-worry group, a broad known-person harm group and domain-specific worry combinations. The depth-2 burden tree has only 29.7% balanced accuracy. However, agreement with existing Ward8 falls to .7399, from approximately .8201 for E8. The earlier grid-selected eight-cluster alternative had Ward8 ARI .9010 but was not one of the two strongest pilots in this extension, so it was not reconfirmed on the new samples. There is a real tradeoff between this stability objective, cross-method agreement and profile meaning. Ward uses its original geometry, so this comparison must not be described as a new matched-geometry test.

Newly supported fine resolution. T9 passes at score .8129, only .0129 above the working threshold. It includes an additional strongly food-worried group (7.8%). A food-worry rule has recall 99.8% but precision 36.5%: food worry is shared with other groups and does not uniquely define this cluster. Sampled neighbouring settings have substantially different partitions and pilot stability. This is not evidence of a broad stable parameter plateau, and a ninth cluster must earn its place through a useful substantive distinction.

Why fresh confirmation matters. For k=5, the highest pilot score (.9332) fell to .6486 and failed, whereas the other shortlisted setting confirmed at .9131. Likewise, the highest k=8 pilot did not have the strongest confirmation score. T7 remains exactly the previously selected reference partition (reference ARI 1), despite a changed confirmation score. Unchanged B4 scored .8559 in the previous confirmation and .9145 here. Different resampling draws and initialization seeds can materially change these finite-repetition summaries. Compare B/E and tuned models within the same confirmation batch; do not interpret a score change across batches as a change in the frozen model or an improvement caused by tuning.

Neighbourhood evidence is descriptive. The saved table contains the nearest already tested parameter settings, their pilot stability and their reference ARI to the selected partition. It is not a complete local surface or a new automatic plateau threshold. Different k can have different geometries; comparisons are approximate refinements, not a strict lattice.

Decision: retain original B4/E8 as frozen references until explicit scientific review. Numerical eligibility is necessary under this protocol, not sufficient for adoption. Review profile meaning in original response units, single-feature dependence and nearby settings before changing downstream assignments. Do not select k solely by the highest stability score or treat the tuned worry coding as an estimated psychological scale.

Limitations: these are the same development countries; fresh resampling is not independent external-country validation. Earlier held-out findings were already known. Pairwise ARIs share fitted partitions and are not independent observations. Comparisons with earlier tuning scores use different confirmation samples, except where the unchanged B/E benchmarks were explicitly rerun on the new common samples. Frozen models, respondent IDs, handoff assignments, held-out results, maps, Week 6 and the manuscript remain unchanged.

## 3. Evaluate profile meaning, individual stability and additional distinctions

**Objective.** Evaluate steps 1–3 agreed after the adaptive search, using only
existing fits. Compare B4/E8 with the selected k=3,...,9 alternatives. Do not
run more optimization, change the eligibility rules or use Ward as a selection
requirement in this round. No geographical or held-out validation is added.

These are nine reference partitions and 54 profiles on the same 109,488
complete-response development respondents. B4/E8 refer to the original frozen
B/E models. Names such as T5 C2 are analysis keys, not new frozen assignment IDs.
All frozen models, IDs and downstream assignments remain unchanged.

In [ ]:
PROFILE_EVALUATION = ROOT / 'course_artifacts/week04_kmeans_profile_evaluation/59d97cff396076a4'
display(pd.Series(json.loads((PROFILE_EVALUATION / 'completion.json').read_text())))

### 3.1. Step 1: compare meanings in the original survey responses

The profile cards show both equal-country-weighted prevalence and the ordinary
respondent proportion. Original worry remains 0=not, 1=somewhat, 2=very worried;
the three categories are shown separately in the stacked bars. The heatmap uses
very-worried prevalence and separate personal/known-person harm prevalence,
all on a common 0–100% scale. It is not a heatmap of transformed centroids.

The full category table includes within-profile prevalence, outside-profile
prevalence and their difference. The concise anchors use distinct features,
not several categories of the same feature. Descriptive names are provisional.
The previous one-feature and depth-1/2/3/5 burden-tree summaries are reused,
not fitted again. Precision and recall must be read together.

In [ ]:
profile_cards = pd.read_csv(PROFILE_EVALUATION / 'profile_review_cards.csv')
display(profile_cards[['model', 'cluster', 'description', 'weighted_share', 'respondent_share',
                       'mean_worry_0_to_2', 'mean_personal_harm_prevalence',
                       'mean_known_person_harm_prevalence']].round(3))
from IPython.display import Image
display(Image(filename=str(PROFILE_EVALUATION / 'original_unit_profile_heatmap.png')))
display(Image(filename=str(PROFILE_EVALUATION / 'original_worry_category_distributions.png')))

### 3.2. Step 2: does each fixed reference profile recur?

An overall pairwise ARI can conceal an unstable small profile or a reference
partition that is less representative than its resampled alternatives.
For each model, reuse 50 respondent-resampling and 50 country-resampling fits.
Every fit has already assigned the same full development population.

Let $C_a$ be reference cluster $a$, $C'_b$ a cluster in a resampled fit, and
$w(A)=\sum_{i\in A}w_i$ with fixed full-development equal-country weights.
Set $M_{ab}=w(C_a\cap C'_b)$. Choose a one-to-one label match $\pi$ maximizing
$\sum_a M_{a,\pi(a)}$. For each matched cluster compute

$$P_a=\frac{M_{a,\pi(a)}}{w(C'_{\pi(a)})},\qquad
R_a=\frac{M_{a,\pi(a)}}{w(C_a)},\qquad
J_a=\frac{M_{a,\pi(a)}}{w(C_a)+w(C'_{\pi(a)})-M_{a,\pi(a)}}.$$

Precision describes the purity of the matched resampled group relative to
the reference group. Recall describes how much of the reference group remains
together. Jaccard combines their intersection and union. Also report
$J_a^{best}=\max_b w(C_a\cap C'_b)/w(C_a\cup C'_b)$, without a one-to-one
constraint. This reveals matching artefacts but allows multiple reference
groups to choose the same resampled group; it cannot by itself prove that the
profiles remain distinct.

Report median, lower decile and minimum across **50 reference-versus-resample
comparisons per test and cluster**. These are not the 1,225 pairwise ARIs used
for the original retention rule. Jaccard is not ARI: **do not transfer the
0.80 ARI cutoff to this new diagnostic**. The original numerical eligibility
flags are preserved; these additional measures inform scientific review.

Reference diagnostics also compare the saved reference to each repetition
using ordinary unweighted ARI. Within each test, the cached fit with largest
mean ARI to the other 49 fits is called the ARI medoid. It is only a diagnostic
representative; no medoid has replaced a reference.

For a further numerical check, evaluate each saved set of centres on the
same full development data and weights:

$$L^{(r)}=\sum_i w_i\min_c\|X_i-c_c^{(r)}\|^2.$$

Compare $L^{(r)}$ with the saved full-data reference objective **only within
the same geometry**. A lower value proves that the reference is not the
lowest-inertia solution among these saved centres. It does not establish a
global optimum or a scientifically better profile structure. This check uses
saved centres and assignments; it performs no refitting.

Cluster-wise overlap follows the general principle of
[Hennig's cluster-stability assessment](https://www.sciencedirect.com/science/article/pii/S0167947306004622),
adapted here to fixed equal-country weights and our two existing resampling tests.

In [ ]:
review = pd.read_csv(PROFILE_EVALUATION / 'model_review_summary.csv')
display(review[['model', 'k', 'score', 'respondents_min_cluster_jaccard_p10',
                'countries_min_cluster_jaccard_p10', 'respondents_minimum_best_any_jaccard_p10',
                'countries_minimum_best_any_jaccard_p10', 'cached_centres_with_lower_full_objective']].round(3))
display(Image(filename=str(PROFILE_EVALUATION / 'individual_profile_stability.png')))
for filename in ['reference_representativeness.csv', 'reference_inertia_diagnostic.csv']:
    print(filename)
    display(pd.read_csv(PROFILE_EVALUATION / filename).round(4))

### 3.3. Step 3: what does each additional profile contribute?

Compare fixed reference partitions through weighted membership overlaps and
original-response differences. In the figure, each target-cluster column sums
to 100% and shows which source profiles its members came from. A target group
can cut across several source groups. Different k also use different optimized
geometries, so these are approximate membership relationships, not a strict
lattice or proof that each extra group is a split of one earlier group.

The largest-overlap source comparison reports its fraction of each target group
and the changes in original category proportions. Mean feature total variation
is shown separately for worry, personal harm and known-person harm. These are
descriptive differences, not significance tests or another selection score.
A high aggregate overlap must not hide a poorly recovered or cross-cutting
additional profile.

In [ ]:
display(Image(filename=str(PROFILE_EVALUATION / 'cross_resolution_membership_overlaps.png')))
pair_comparisons = pd.read_csv(PROFILE_EVALUATION / 'partition_comparisons.csv')
display(pair_comparisons.loc[(pair_comparisons.source.isin(['B4', 'E8', 'T5', 'T7', 'T8'])) &
                            (pair_comparisons.target.isin(['T4', 'T5', 'E8', 'T7', 'T8', 'T9']))].round(3))
added_profiles = pd.read_csv(PROFILE_EVALUATION / 'added_profile_review.csv')
display(added_profiles.loc[added_profiles.source.eq('T8') & added_profiles.target.eq('T9')].round(3))

### 3.4. Critical interpretation and current decision

Scope: compare the seven numerically eligible adaptive selections with unchanged B4/E8. These are nine fixed reference partitions and 54 profiles. All 900 existing confirmation fits are reused. No K-means, explanatory classifier, Ward, geographic or held-out model is fitted. The original ARI/structural requirements remain unchanged; no Jaccard rejection threshold is introduced.

Meaning of the new diagnostic: after one-to-one maximum-overlap label matching, precision measures how much of the matched resampled group comes from the reference group; recall measures how much of the reference group remains in that matched group. Weighted Jaccard is their intersection mass divided by union mass. The weights are the fixed full-development equal-country weights. Each test has 50 reference-versus-resample values per cluster, not 1,225 independent values. The best-any Jaccard allows every reference profile to select its own best-matching resampled group without a one-to-one constraint. It diagnoses whether an apparent weakness is caused by the global assignment; it can map several reference groups to the same resampled group and therefore does not establish that all profiles remain distinct.

T3: minimum cluster lower-decile matched Jaccard is 0.991 in the respondent test and 0.989 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.991 and 0.989. 0/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

B4: minimum cluster lower-decile matched Jaccard is 0.946 in the respondent test and 0.928 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.946 and 0.928. 0/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T4: minimum cluster lower-decile matched Jaccard is 0.984 in the respondent test and 0.958 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.984 and 0.958. 0/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T5: minimum cluster lower-decile matched Jaccard is 0.957 in the respondent test and 0.943 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.957 and 0.943. 0/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T6: minimum cluster lower-decile matched Jaccard is 0.418 in the respondent test and 0.419 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.422 and 0.422. 57/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T7: minimum cluster lower-decile matched Jaccard is 0.452 in the respondent test and 0.452 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.452 and 0.452. 88/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

E8: minimum cluster lower-decile matched Jaccard is 0.085 in the respondent test and 0.087 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.414 and 0.414. 57/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T8: minimum cluster lower-decile matched Jaccard is 0.524 in the respondent test and 0.471 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.524 and 0.471. 0/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

T9: minimum cluster lower-decile matched Jaccard is 0.081 in the respondent test and 0.066 in the country test. With best-any matching, the corresponding minimum cluster lower deciles are 0.086 and 0.086. 32/100 cached sets of centres have lower full-population inertia than the reference, comparing only identical geometry and weights within this model.

Substantive comparison: T3 is very reproducible, but its no-crime-worry group is almost reconstructed by that single response (balanced accuracy 97.7%, precision 92.8%). Its high stability does not by itself make it an informative multidomain solution. B4, T4 and T5 recover all their reference profiles much more consistently than the finer models. T4 is a different family from B4 (ARI .332), with a broad-harm group and a known-person traffic-harm group. The latter has a necessary traffic-harm marker but only 67.4% single-feature precision, so it is not uniquely defined by that feature.

T5 is a particularly useful intermediate candidate: lower worry (36.3%), broad known-person harm (13.8%), stronger crime/weather worry (18.5%), stronger mental-health/traffic/weather worry (15.8%), and stronger water/food worry (15.5%). Its minimum cluster lower-decile Jaccard is about .957 for respondent resampling and .943 for country resampling. The strongest single-feature rule is less reconstructive than T3: balanced accuracy 88.3%, precision 67.4%, recall 84.3%. Its depth-2 burden tree has only 46.0% balanced accuracy. This supports substantive review of T5 as an alternative, not automatic replacement of the frozen models.

Fine-resolution caution: T6 and T7 pass the original pairwise-ARI requirements, yet their crime/weather distinctions do not consistently recover the fixed reference groups. In T7, country-resampled partitions have median pairwise ARI .985, while their median ARI to the saved full-data reference is only .876. Thus the saved reference is less representative of the resampling ensemble than a pairwise summary alone suggests. The selected within-test medoids are diagnostics only; they have not replaced any reference partition.

E8 has a specific weakness in its crime-and-traffic group (8.1% equal-country prevalence). Lower-decile matched Jaccard is only .085/.087 in the respondent/country tests. Part of this extreme value comes from the one-to-one matching constraint: best-any lower deciles are about .414/.414, still far from complete recovery. The group and its boundary with other crime/traffic profiles are less secure than the overall ARI suggests. This qualifies the robustness claim for that fine distinction; it does not erase all useful E8 structure or invalidate the already frozen held-out assignments.

T8 is the more promising fine-resolution alternative in this review, but it is not uniformly stable. Most reference groups have very high median overlap. Weather/crime boundaries occasionally change: minimum cluster lower-decile Jaccard is .524/.471 in the two tests. The low-worry, broad-harm and several other profile distinctions recover much more consistently. Report the reliable backbone and these sensitive boundaries separately, rather than describing all eight profiles as equally robust.

The ninth profile is not supported as a consistently recovered fixed group. T9 passes overall ARI, but its added food-worry group (7.8%) has median matched Jaccard about .089. Even best-any lower-decile Jaccard is only about .086 in each test. This is not merely a label permutation problem. Although 99.8% of its reference members are very worried about food, that category purity does not establish reproducible joint cluster membership. Treat the ninth distinction as exploratory and fragile, not a validated extra profile.

What the saved objective values show: some resampled sets of centres outperform the full-data reference on the same full-population weighted K-means objective. This occurs for 57/100 E8 fits, 57/100 T6 fits, 88/100 T7 fits and 32/100 T9 fits; the largest relative reductions are small (roughly .06%–.18%) but can accompany substantial membership changes. This directly demonstrates that these references are not the lowest-inertia solutions among the saved centres. It is consistent with competing near-optimal solutions and initialization sensitivity; it does not prove a unique global optimum, nor make the lower-inertia partition scientifically correct. No analogous improvement is found among the 100 saved fits for B4, T3, T4, T5 or T8. Raw inertia is never used to rank different geometries.

Additional clusters are not automatically nested splits. T8 and T9 have ARI .870 and 91.6% aggregate largest-source overlap, but only about 23.5% of the T9 food group comes from its largest single T8 source group. It cuts across existing groups rather than splitting just one. Likewise, T5-to-T8 aggregate purity is about 82.5%, but one fine group has no majority T5 source. Use the complete overlap table and original-response differences, not a strict lattice or a simple finer-is-better narrative.

Current decision: keep B4/E8 and all downstream assignments unchanged. B4 remains a defensible coarse benchmark. T4 and especially T5 merit substantive comparison as alternative coarse/intermediate descriptions. T8 merits review as a fine-resolution alternative with explicit uncertainty around particular boundaries. T6/T7 reference representativeness and the T9 extra group require caution before adoption. Ward agreement is not used as a selection requirement in this round. No numerical eligibility rule was changed after seeing these results.

Limits: this is a post-search development-only assessment, not an independent replication or a new held-out validation. Matching is based on a fixed reference and can be affected by merges/splits; best-any matching and reference-representativeness diagnostics are provided to expose that limitation. The within-test ARI medoid is only the most representative cached fit under that particular ARI criterion. These summaries do not separate all sources of sampling and solver variability, and no new cluster is frozen.

### 3.5. Reproduce these diagnostics without fitting models

Leave the switch off to read saved results. The evaluation checks fingerprints
of the selected references and all 900 fit caches. The separate numerical
diagnosis only evaluates saved centres; it does not call K-means or a classifier.
All code uses the development cohort and preserves earlier results and user edits.

Implementation: [profile and membership calculations](../.week04_parameter_search/evaluate_profiles.py),
[reference diagnosis](../.week04_parameter_search/diagnose_reference_profiles.py),
[figures and interpretation](../.week04_parameter_search/summarize_profile_evaluation.py),
[verification](../.week04_parameter_search/test_profile_evaluation.py).

In [ ]:
RUN_CACHED_PROFILE_EVALUATION = False
if RUN_CACHED_PROFILE_EVALUATION:
    import subprocess
    import sys
    for filename in ['evaluate_profiles.py', 'diagnose_reference_profiles.py',
                     'summarize_profile_evaluation.py', 'test_profile_evaluation.py']:
        subprocess.run([sys.executable, str(ROOT / '.week04_parameter_search' / filename)], check=True)
print(json.loads((PROFILE_EVALUATION / 'verification.json').read_text()))

## 4. Select T5 and check shallow explanations — 16 October 2026

**Objective.** Decide which development model to take forward after Section 3,
then check whether its membership is almost a simple rule or a burden ranking.
This updates the earlier decision to keep B4/E8 as the main pair. Earlier
sections are the historical research record; their saved results are not changed.

**Decision: take T5 forward as the primary model.** It passes all original
structural and median/lower-decile ARI criteria. Its weakest lower-decile ARI
is 0.9131, close numerically to B4's 0.9145 on the same confirmation batch.
Every T5 reference profile recurs strongly: the smallest country-test
lower-decile matched Jaccard is 0.9427. These are different diagnostics, not
interchangeable measures or evidence of statistical equivalence.

T5 retains recognizable lower-worry, broad known-person-harm, crime/weather
and water/food patterns, and adds a useful mental-health/traffic/weather
distinction. It is not exactly B4 plus one split: the geometry also changes.
Its approximate reference agreement with B4 is ARI 0.5693.

The exact selected parameters are $k=5$, $q=0.41670089020376244$,
$t=1/q=2.399802888616365$, and $\alpha=0.5837857862168715$.
Original worry categories become $0,q,1$. Subtract each respondent's six-feature
worry mean, append that mean, and apply the existing reduced-mean geometry.
The fourteen harm indicators remain separate. This is a selected modelling
geometry, not an estimate of a uniquely correct psychological scale.

B4 is still a defensible historical benchmark, not a failed model. T4 offers
a stable but different harm/traffic-oriented typology. T3 is too close to a
single-feature classification for this scientific aim. T6/T7 have reference
representativeness concerns. E8/T9 have weak individual profiles. T8 remains
an exploratory finer alternative, not a co-equal primary model. Archive these
results; do not erase them. Do not require a new Ward comparison in this round.

### 4.1. What did the two existing reconstruction checks establish?

The **burden trees** predict all five fixed cluster labels from only two scores:
mean original 0/1/2 worry and mean combined respondent-or-known-person harm.
They use country-grouped five-fold cross-validation, all five folds, trees of
depth 1, 2, 3 and 5, minimum leaf size 200, and equal-country weights in training
and evaluation. These scores are not the optimized clustering coordinates.

The **single-feature check** predicts membership in each cluster versus outside
it. On training countries, search all 20 original features, every category
threshold and both directions; maximize weighted balanced accuracy. Evaluate
the chosen rule on that fold's evaluation countries. These are custom
balanced-accuracy stumps, not ordinary Gini trees. The cohort has complete
responses, so no imputation or feature-specific exclusion is needed.

Both checks reuse fixed development labels: they describe fidelity to an
already learned partition, not independent recovery of clusters. Low burden
reconstruction does not exclude a simple rule involving several features.

In [ ]:
T5_EXPLANATIONS = ROOT / 'course_artifacts/week04_t5_shallow_explanations/4f0f385fb4cbe644'
display(pd.read_csv(T5_EXPLANATIONS / 'burden_tree_summary.csv').round(4))
display(pd.read_csv(T5_EXPLANATIONS / 'single_feature_summary.csv').round(4))

### 4.2. Additional check: ordinary shallow trees with all 20 features

**Objective.** Check the missing possibility: can a small multifeature tree
almost reproduce T5? Use the same five country folds and original features.
For each depth 1, 2, 3 and 5, fit
`DecisionTreeClassifier(max_depth=depth, min_samples_leaf=200, random_state=2023)`.
Use its default Gini criterion, no class reweighting, and equal-country sample
weights. Fit on four folds, evaluate on the fifth; every country is evaluated
once. Do not recluster respondents. The saved fold rules and weighted confusion
matrices make the behaviour inspectable; no extra full-development tree is fit.

Balanced accuracy is mean recall over the five classes. A depth-1 tree has at
most two leaves, so its five-class balanced-accuracy ceiling is 40%; depth 2
has a ceiling of 80%. Their limited capacity must be acknowledged when reading
low scores. These are ordinary greedy trees, not a search over every possible
rule set. The original labels used all development countries, so holding out
tree-evaluation countries does not turn this into independent clustering validation.

In [ ]:
display(pd.read_csv(T5_EXPLANATIONS / 'all_feature_tree_summary.csv').round(4))
t5_tree_profiles = pd.read_csv(T5_EXPLANATIONS / 'all_feature_tree_per_cluster_summary.csv')
display(t5_tree_profiles.loc[t5_tree_profiles.depth.isin([2, 3])].round(4))
print('Example: depth-3 tree fitted without the first evaluation-country fold')
print((T5_EXPLANATIONS / 'depth3_fold1_rules.txt').read_text())

### 4.3. Interpretation and next step

Burden-tree balanced accuracy at depths 1/2/3 is 32.7% / 46.0% / 51.2%.
All-feature trees reach 36.8% / 54.0% / 67.9%; depth 5 reaches 73.7%.
At depth 3, recall for the broad known-person-harm profile is only 19.3%.
These trees capture interpretable worry patterns but do not almost reproduce
the five-profile solution. The strongest single-feature rule, water worry,
has 88.3% balanced accuracy, 67.4% precision and 84.3% recall. It is informative
but not a near-unique definition of that group. No new automatic rejection
threshold is imposed, and interpretability itself is not a defect.

This supports proceeding with T5; it does not prove that the profiles are
natural latent types or irreducible to every possible simpler description.
Freeze the existing development centres without refitting, then assess their
unchanged assignment in the 29 previously held-out countries. Their earlier
B/E results are already known: this is **secondary transfer assessment**, not
a newly untouched external test. See Section 5 below.

Implementation: [shallow-tree check](../.week04_parameter_search/check_t5_shallow_trees.py).
The old B/E files and downstream results remain preserved.

## 5. T5 freeze and secondary transfer assessment — 16 October 2026

**Objective.** Apply the primary T5 model selected after adaptive development
search and profile-level review. Appendix A replays the original B/E
analysis. This section records the new decision separately; it does not
retroactively change the earlier experiments, seals, assignments or findings.

T5 uses $k=5$, $t=2.399802888616365$, $q=1/t=0.41670089020376244$ and
$\alpha=0.5837857862168715$. Six respondent-centred worry coordinates and
their appended mean are multiplied by $\sqrt{\alpha/6}$; fourteen separate
harm indicators by $\sqrt{(1-\alpha)/14}$. The implementation retains the
original operation order. There is no imputation, z-standardization or PCA.
See Section 4 above for the shallow-tree and model-selection evidence.

**Validation status matters.** These countries were originally held out for
B/E, but those results were already examined before the new search. This is
secondary assessment, not pristine external confirmation. All new tuning
used development countries; no T5 fitting or tuning uses held-out responses.

In [ ]:
T5_TRANSFER = ROOT / 'course_artifacts/week05_t5_secondary_transfer/832b01713bc6b240'
T5_PACKAGE = T5_TRANSFER / 'frozen_model'
t5_seal = json.loads((T5_PACKAGE / 'freeze_seal.json').read_text())
for filename, expected in t5_seal['files'].items():
    assert hashlib.sha256((T5_PACKAGE / filename).read_bytes()).hexdigest() == expected
t5_specification = json.loads((T5_PACKAGE / 'specification.json').read_text())
display(pd.Series(t5_specification['spec']))
display(pd.Series(json.loads((T5_TRANSFER / 'transfer_complete.json').read_text())).drop('files'))

### 5.1. Freeze exact centres, profile IDs, thresholds and descriptive anchors

The five stored development centres are copied without a K-means refit and
reproduce all 109,488 reference assignments exactly. The seal is written
before this pipeline reads T5 held-out responses. It explicitly states that
the project had previously examined these countries for B/E.

Keep IDs **T5_C0,...,T5_C4** in stored centre order; do not order them by
prevalence or reuse B/E cluster numbers. Names are descriptive shorthands,
not exhaustive or mutually exclusive response definitions. Freeze three
largest positive category-versus-outside contrasts per profile, using distinct
features, and each assigned profile's weighted development distance p95.

Assignment is $\hat c_i=\arg\min_c\|x_i-\mu_c\|_2$, with the first stored
centre used on an exact tie. Margin is $(d_{i,2}-d_{i,1})/d_{i,2}$, where
$d_{i,1},d_{i,2}$ are the smallest two distances. A distance flag means
$d_{i,1}$ exceeds the frozen development p95 for $\hat c_i$; it does not
exclude the respondent or measure a misclassification probability.

Apply exactly these rules to 27,972 complete respondents in all 29 held-out
countries. Retain the other 2,118 respondents in the eligibility file, with
missing T5 labels. Do not impute an assignment or change any model parameter.

### 5.2. Compare prevalence, original meanings and representation quality

Equal-country weights preserve relative survey weights within each complete-case
country, then give each country equal mass within the displayed area. Respondent
shares are also saved. Prevalence need not match development prevalence.

For original feature $j$, compare the complete category distributions within
profile $c$ using $V_{cj}=\tfrac12\sum_a|p^{held}_{cja}-p^{dev}_{cja}|$.
Report both mean and maximum feature total variation. These are descriptive
category shifts, not centroid distances, ARI or a new transfer cutoff. Check
the development-selected anchors' direction against outside-profile responses.

The intervals use 1,000 uniform country bootstrap draws with replacement,
preserving within-country survey weights and fixing centres and assignments.
They reflect heterogeneity over the observed 29 countries, not global sampling,
model-selection/fitting uncertainty, or the missing responses.

In [ ]:
display(pd.read_csv(T5_TRANSFER / 'profile_transfer_summary.csv').round(4))
from IPython.display import Image
display(Image(filename=str(T5_TRANSFER / 't5_prevalence_transfer.png')))
display(Image(filename=str(T5_TRANSFER / 't5_original_response_transfer.png')))
t5_anchors = pd.read_csv(T5_TRANSFER / 'heldout_frozen_anchor_comparison.csv')
display(t5_anchors.loc[t5_anchors.scope.eq('global'),
    ['cluster', 'feature', 'category', 'development_proportion', 'proportion',
     'development_contrast', 'difference_vs_outside', 'heldout_contrast_direction_preserved']].round(4))
display(pd.read_csv(T5_TRANSFER / 'heldout_vs_development_quality.csv').round(4))

### 5.3. Inspect country problems, not only aggregate averages

Display distance flags together with complete-response retention. Country/profile
semantic tables also give respondent counts and Kish effective sample sizes.
Small conditional groups should not be interpreted as precisely measured
country-specific profile definitions. Missing responses can cause selective
exclusion; these results describe complete respondents, not all adults without
qualification. No new regional clustering is performed for T5 in this task.

In [ ]:
t5_country = pd.read_csv(T5_TRANSFER / 'heldout_country_quality_and_retention.csv')
display(t5_country.sort_values('profile_p95_exceedance', ascending=False)[
    ['Country', 'prepared', 'complete', 'retained_fraction', 'profile_p95_exceedance',
     'distance_median', 'margin_median']].round(4))
t5_country_profiles = pd.read_csv(T5_TRANSFER / 'heldout_profile_semantic_differences.csv')
display(t5_country_profiles.loc[t5_country_profiles.scope.eq('country')].sort_values(
    'mean_feature_total_variation', ascending=False).head(15).round(4))

### 5.4. Interpretation and decision

Decision and scope. T5 is the approved primary model to take forward. The exact development reference is sealed, not refitted: k=5, q=0.41670089020376244, t=2.399802888616365, alpha=0.5837857862168715. T5_C0,...,T5_C4 preserve its stored centre order. B4/E8 and every historical result remain available as benchmarks. The existing maps, manuscript, publication package and Week 6 have not yet been migrated.

Shallow explanations. All five country-grouped folds were used. With the two original-unit burden scores, depths 1/2/3 achieved mean balanced accuracies 32.7%, 46.0% and 51.2%. The additional trees supplied with all 20 original features achieved 36.8%, 54.0% and 67.9%; depth 5 reached 73.7%. At depth 3, broad known-person-harm recall was only 19.3%, despite high recall for lower worry. These trees capture useful patterns but do not nearly reconstruct the entire typology. The strongest single-feature rule uses water worry, with balanced accuracy 88.3%, precision 67.4% and recall 84.3%. This is an informative marker, not a near-unique cluster definition. A depth-1 multiclass tree has at most two leaves and depth 2 at most four: for five nonempty classes their balanced-accuracy ceilings are 40% and 80%. Low shallow scores alone therefore cannot prove complexity. No new automatic rejection cutoff was introduced. Ordinary greedy Gini trees were used; this does not rule out every possible optimized rule set or establish that the profiles are natural latent types. The fixed cluster labels were originally learned using all development countries, so this cross-validation assesses explanatory fidelity, not fully independent clustering generalization.

Transfer population. Apply the sealed centres to 27,972 complete respondents from all 29 held-out countries. The other 2,118 of 30,090 respondents remain unassigned because of missing profile responses. A separate ID-linked T5 handoff includes all 146,910 prepared respondents: 137,460 assigned and 9,450 excluded. Do not fill excluded labels, join by row order, or substitute T5 cluster numbers into B/E columns.

Prevalence. Development to held-out equal-country shares are: Lower worry 36.3% to 35.9%; Broad known-person harm 13.8% to 12.5%; Crime/weather worry 18.5% to 18.3%; Mental-health/traffic/weather worry 15.8% to 17.9%; Water/food worry 15.5% to 15.4%. Different prevalence is allowed and was not used to retune the model.

Profile identity. All 15/15 development-selected anchor directions remain positive in aggregate held-out data. Mean feature total variation within profiles ranges from 0.0083 to 0.0281; the largest single-feature difference is 0.0554. This measures original-category distribution shifts, not centroid distances, ARI, or a newly defined pass/fail score. Semantic retention partly follows from assigning people using these same features; it supports portability of this fixed description, not independent discovery of the same groups.

Assignment quality. Overall median distance changes by a factor 1.0048; median assignment margin changes from 0.1515 to 0.1524. The fraction beyond the assigned-profile development distance p95 changes from 4.95% to 5.69%. Flags identify less typical assigned responses, not unassigned cases, misclassification probabilities or an overall rejection rule.

Country limitations. The largest distance-flag fractions are Sierra Leone 18.8% (complete-response retention 78.7%); Namibia 16.9% (complete-response retention 94.4%); Somalia 16.7% (complete-response retention 82.9%). Global averages conceal these weaknesses. Country/profile tables include respondent counts and Kish weighted effective sizes; very small groups have uncertain conditional descriptions. For example, Sweden's water/food group contains only 15 respondents (Kish effective size 9.8), so its mean feature total variation of .200 is poorly estimated. Somalia's water/food group has mean feature total variation .225 with 173 respondents (effective size 138.8): this is a substantive country-specific shift that cannot simply be dismissed as a 15-person group. These examples qualify any claim that all five profile meanings reproduce equally well in every country. Complete-case selection may change apparent country profiles; the assessment does not establish representativeness of respondents excluded for missing answers.

Uncertainty and conclusion. The 1,000 country bootstrap intervals keep the model fixed and preserve relative survey weights within each sampled country. They describe variation over these 29 observed countries, not all-world sampling, respondent nonresponse, model fitting or parameter-selection uncertainty. The results support broad aggregate descriptive transfer of T5, with material country-specific qualifications. No automatic numerical transfer criterion had been specified; this is an analytical synthesis using the same diagnostic framework as the earlier B/E assessment. The held-out countries were previously examined for B/E and the new search was undertaken after those results were known. This must be reported as secondary transfer assessment, not pristine external confirmation. No new local-region clustering, independent held-out clustering or all-data refit was performed. T8 remains exploratory, not a co-equal main model.

### 5.5. ID-linked handoff and reproducibility

The separate `all_respondents_T5_eligibility_and_frozen_profiles.parquet` retains
146,910 unique IDs, eligibility and reasons, split, geography, survey weights,
T5 profile ID/name, distances, margin and flag. Its 137,460 complete cases have
assignments; 9,450 incomplete cases do not. Join on `WPID_RANDOM`, never row
position. Use the split-specific equal-country weight for split comparisons,
or the all-country weight for pooled summaries. Do not concatenate separately
normalized split weights unchanged.

**Downstream status:** T5 is the new primary scientific choice; the earlier B/E
handoff is preserved. The local GitHub publication now defaults to T5 in its
profile explorer and supplies a readable T5 model and aggregate evaluation.
Country-distance pages retain explicitly labelled historical B/E metrics; the
paper and Week 6 remain unchanged. No all-data or held-out clustering is refitted.

The completed transfer is fingerprint-checked and not repeated on rerun.
Verification independently reconstructed held-out assignments, checked category
probabilities and weights, preserved every old B/E model hash and retained all
IDs. A final console-summary error after scientific completion was repaired
without new assignments or fits; the original table, executed source and
reporting incident are preserved in the results folder.

Implementation: [freeze and transfer](../.week05_build/t5_secondary_transfer.py),
[reporting](../.week05_build/report_t5_secondary_transfer.py),
[verification](../.week05_build/test_t5_secondary_transfer.py).

In [ ]:
print(json.loads((T5_TRANSFER / 'verification.json').read_text()))
display(pd.Series(json.loads((T5_TRANSFER / 'primary_model_decision.json').read_text())))
# Reuse by ID, not by row position; the original survey dataset is not overwritten.
t5_handoff = pd.read_parquet(T5_TRANSFER / 'all_respondents_T5_eligibility_and_frozen_profiles.parquet')
assert t5_handoff.WPID_RANDOM.is_unique
display(t5_handoff.groupby('country_split').agg(
    prepared=('WPID_RANDOM', 'size'), complete_and_assigned=('complete_case', 'sum')))

### 5.6. Evaluation completed and local publication prepared

All agreed T5 checks are complete: numerical confirmation, individual-profile
recovery, original-unit meaning, explanatory checks, exact development freeze
and secondary held-out transfer. T5 remains the primary five-profile description,
with country-specific and complete-case limitations. There is no automatic
held-out rejection threshold, and no all-data refit or new Ward experiment.

The local GitHub package includes the updated weekly workflow, search and
evaluation aggregates, readable T5 centres and assignment code, and an explorer
that starts with T5 while preserving B/E comparisons. The public assignment code
reproduces all 27,972 held-out T5 labels. Browser controls, three widths, both
themes, offline operation, privacy and guide links were checked. No respondent
records or assignment files are published; no commit or push was performed.

Country-distance maps still represent historical B/E allocations, explicitly
labelled. They must not be described as T5-specific distances. The Week 6 B/E
prediction draft and the manuscript have not been migrated by this task.

The finalization record is `evaluation_finalization.json` in the T5 results
folder; the original freeze seal and all historical B/E evidence are preserved.

## Appendix A. Original B/E analyses — preserved historical record

These results precede the new parameter search in the actual research history.
They show regional fits, transfer, country comparisons and the original B/E
handoff. They are retained for comparison and reproducibility, not presented as
new T5 results. Existing maps and Week 6 still use that B/E handoff until a
separate migration is implemented.


### A.1. Verify the frozen definitions before using held-out responses

    B = balanced 0/1/2, k4; E = worry-emphasized 0/1/3, k8. The seal records
    that these definitions were frozen before opening held-out responses.
    This notebook replays the completed validation. It does not claim the held-out
    data are still unopened, and does not refit on all countries afterwards.
    Keep all original cluster IDs and profile names for later objectives.


In [ ]:
FROZEN = ART / 'week05_frozen_objective1_v1/123d248adda6115f'
HELDOUT = ART / 'week05_heldout_transfer_v1/123d248adda6115f'
REGIONAL = ART / 'week05_regional_validation_v3/7c24b461bbb895b5'
LOCAL = ART / 'week05_local_resolutions_v4/f3145c704f52c156'
HANDOFF = ART / 'week05_objective1_handoff_v1/123d248adda6115f'
seal = json.loads((FROZEN / 'freeze_seal.json').read_text())
assert seal['status'] == 'frozen_before_heldout_responses'
for filename, digest in seal['files'].items():
    assert hashlib.sha256((FROZEN / filename).read_bytes()).hexdigest() == digest, filename
bundle = joblib.load(FROZEN / 'models.joblib')
assert bundle['context'] == seal['context']
assert bundle['feature_order'] == FEATURES
for name, model in bundle['models'].items():
    print(name, model['spec'])
    display(pd.Series(model['profile_names'], name='profile_name').rename_axis('cluster'))


### A.2. Assign complete respondents to the fixed centroids

    Derive twenty original-unit features. A missing response is not a zero; do
    not impute into this final specification. Assign only complete responses.
    Distances use the frozen 21-coordinate geometry. Exact ties go to the first
    stored centre, preserving IDs. A distance flag means farther than that
    profile's weighted development 95th percentile: it is not a rejection or a
    probability. The margin is geometric separation from the second-nearest
    centroid, not confidence in a true label.


In [ ]:
prepared = pd.read_parquet(ART / 'week03_2023_analysis_data.parquet')
for feature, source in zip(HARM, ['WP22442', 'WP22443', 'WP22444', 'WP22445', 'WP22446', 'WP22447', 'WP22448']):
    values = prepared[source]
    assert values.dropna().isin([1, 2, 3, 4, 98, 99]).all()
    prepared[feature + '_personal'] = values.map({1: 1., 2: 0., 3: 1., 4: 0.})
    prepared[feature + '_known_person'] = values.map({1: 0., 2: 1., 3: 1., 4: 0.})
complete = prepared[FEATURES].notna().all(axis=1)
data = prepared.loc[complete].reset_index(drop=True)
assert data.WPID_RANDOM.is_unique and len(data) == 137460
registry = pd.read_parquet(HANDOFF / 'all_respondents_eligibility_and_frozen_profiles.parquet').set_index('WPID_RANDOM')
assert prepared.WPID_RANDOM.is_unique and set(prepared.WPID_RANDOM) == set(registry.index)
assert np.array_equal(complete, registry.loc[prepared.WPID_RANDOM, 'complete_case'].to_numpy())
assignment = data[['WPID_RANDOM', 'COUNTRY_ISO3', 'GlobalRegion', 'CountryIncomeLevel', 'country_split', 'WGT']].copy()
for name, model in bundle['models'].items():
    spec = model['spec']
    values = data[FEATURES].to_numpy(float)
    worry = np.where(values[:, :6] == 2, spec['top'], values[:, :6]) / spec['top']
    mean = worry.mean(axis=1, keepdims=True)
    X = np.column_stack([(worry - mean) * np.sqrt(spec['alpha'] / 6),
                         mean * np.sqrt(spec['alpha'] / 6),
                         values[:, 6:] * np.sqrt((1 - spec['alpha']) / 14)])
    distance = cdist(X, model['centres'])
    labels = distance.argmin(axis=1)
    nearest = np.sort(distance, axis=1)[:, :2]
    assignment[name + '_cluster'] = labels
    assignment[name + '_distance'] = nearest[:, 0]
    assignment[name + '_margin'] = np.divide(nearest[:, 1] - nearest[:, 0], nearest[:, 1],
                                            out=np.zeros(len(data)), where=nearest[:, 1] > 0)
    assignment[name + '_flag'] = nearest[:, 0] > np.array([model['profile_distance_p95'][int(j)] for j in labels])
    assert np.array_equal(labels, registry.loc[data.WPID_RANDOM, name + '_cluster'].to_numpy())
    assert np.allclose(nearest[:, 0], registry.loc[data.WPID_RANDOM, name + '_nearest_distance'])
print('All 274,920 B/E assignments reproduced exactly; no clustering fit.')
display(prepared.assign(complete=complete).groupby('country_split').complete.agg(['size', 'sum', 'mean']))


### A.3. Describe fixed-global profiles within regions and countries

    “Global” refers to centres fitted jointly on development countries. Applying
    them within each region changes prevalence, not profile definitions. Country
    weights are recalculated within each displayed area. Also show unweighted
    respondent shares. The denominators exclude incomplete respondents, which
    can change each area's composition.


In [ ]:
def prevalence(frame, labels):
    mass = country_weights(frame)
    result = pd.DataFrame({'respondents': pd.Series(labels).value_counts().sort_index(),
                           'equal_country_share': pd.Series(mass).groupby(labels).sum()})
    result['respondent_share'] = result.respondents / len(frame)
    return result.rename_axis('cluster')


In [ ]:
for split, group in assignment.groupby('country_split'):
    for name in ['B', 'E']:
        print(split, name)
        display(prevalence(group, group[name + '_cluster'].to_numpy()).round(3))
regional_rows = []
for (split, region), group in assignment.groupby(['country_split', 'GlobalRegion']):
    for name in ['B', 'E']:
        table = prevalence(group, group[name + '_cluster'].to_numpy()).reset_index()
        table['split'], table['region'], table['candidate'] = split, region, name
        regional_rows.append(table)
regional_prevalence = pd.concat(regional_rows, ignore_index=True)
regional_prevalence.to_csv(OUT / 'week05_recomputed_regional_prevalence.csv', index=False)
display(regional_prevalence.round(3))
# The completed evidence also includes every individual country and original category.
display(pd.read_csv(REGIONAL / 'fixed_global_profile_prevalence.csv').head(24))
display(pd.read_csv(REGIONAL / 'fixed_global_original_profiles_by_region.csv').head(24))


### A.4. Regional refitting tests a different question

    A locally optimized partition need not reproduce a fixed global one. Inspect
    k = 2,...,10 within each region under the fixed B/E representations. Reference
    diagnostics exist for 270 fits. The completed bounded stability search tests
    matched global k, k2/k3, and a normalized-inertia elbow where different.
    It does not test stability for every local k, nor select by local/global ARI.

    Keep the .80 median and lower-decile thresholds. Respondent tests use 50
    within-country 80% samples. Regions with at least five countries use 50
    80%-country samples. With 2–4 countries, leave-one-country-out is diagnostic,
    not the full 50-repeat benchmark; one-country regions cannot test country
    transfer. Stable binary solutions are informative but do not establish the
    richer profile structure we sought. Global structural filters remain a
    descriptive benchmark here, not an extra local selection rule.


In [ ]:
regional_diagnostics = pd.read_csv(LOCAL / 'all_local_resolution_diagnostics.csv')
local_support = pd.read_csv(LOCAL / 'supported_local_resolutions.csv')
display(local_support.round(3))
display(pd.read_csv(LOCAL / 'qualified_local_profile_interpretations.csv'))
display(pd.read_csv(LOCAL / 'regional_supported_k_sets.csv'))


In [ ]:
# Reproduce the bounded extra-k shortlist from inertia, not local/global agreement.
shortlist = []
for (region, name), group in regional_diagnostics.groupby(['region', 'candidate']):
    group = group.sort_values('k').copy()
    inertia = group.inertia.to_numpy()
    group['gap'] = (1 - (group.k - 2) / 8) - (inertia - inertia[-1]) / (inertia[0] - inertia[-1])
    matched_k = bundle['models'][name]['spec']['global_k']
    eligible = group.loc[group.k.between(3, 9) & group.k.ne(matched_k)]
    elbow = int(eligible.loc[eligible.gap.ge(.95 * eligible.gap.max()), 'k'].min())
    shortlist.extend((int(region), name, k) for k in sorted({2, 3, elbow} - {matched_k}))
saved_shortlist = pd.read_csv(LOCAL / 'additional_local_shortlist.csv')
assert set(shortlist) == set(saved_shortlist[['region', 'candidate', 'k']].itertuples(index=False, name=None))


In [ ]:
# An optional direct local fit; it writes nothing to the frozen package.
if RUN_ONE_LOCAL_REFERENCE:
    region, name, k = 1, 'B', 3
    seed = 20261011 + region
    group = data.loc[data.country_split.eq('train') & data.GlobalRegion.eq(region)]
    spec = bundle['models'][name]['spec']
    values = group[FEATURES].to_numpy(float)
    worry = np.where(values[:, :6] == 2, spec['top'], values[:, :6]) / spec['top']
    mean = worry.mean(axis=1, keepdims=True)
    X_local = np.column_stack([(worry - mean) * np.sqrt(spec['alpha']) / np.sqrt(6),
                               mean * np.sqrt(spec['alpha']) / np.sqrt(6),
                               values[:, 6:] * np.sqrt((1 - spec['alpha']) / 2) / np.sqrt(7)])
    with threadpool_limits(limits=1):
        local_model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
        local_model.fit(X_local, sample_weight=country_weights(group))
    print('Illustrative new local fit only; not a replacement for saved local stability tests.')


### A.5. Read held-out transfer against the frozen definitions

    The held-out set has 29 countries, 30,090 respondents; 27,972 have complete
    profile responses. Profile prevalence may differ without invalidating the
    profiles. Compare original response distributions, anchors, assignment
    distances and country-specific problems. There was no preregistered binary
    held-out rejection cutoff; do not invent one now. Saved country-bootstrap
    intervals condition on the fixed profiles, not on discovering new profiles.


In [ ]:
heldout_prevalence = pd.read_csv(HELDOUT / 'heldout_prevalence.csv')
display(heldout_prevalence.loc[heldout_prevalence.scope.eq('global')].round(3))
display(pd.read_csv(HELDOUT / 'heldout_vs_development_prevalence.csv').round(3))
display(pd.read_csv(HELDOUT / 'heldout_frozen_anchor_comparison.csv').round(3))
display(pd.read_csv(HELDOUT / 'heldout_country_bootstrap_intervals.csv').head(30).round(3))
for name in ['B', 'E']:
    flags = []
    for split, group in assignment.groupby('country_split'):
        flags.append({'candidate': name, 'split': split,
                      'weighted_flag_fraction': np.average(group[name + '_flag'], weights=country_weights(group))})
    display(pd.DataFrame(flags).round(4))
display(pd.read_csv(HELDOUT / 'heldout_assignment_quality.csv').head(30).round(3))


In [ ]:
# Recalculate original-category proportions for the global development/held-out profiles.
profile_rows = []
for split, group in data.groupby('country_split'):
    mass = country_weights(group)
    for name in ['B', 'E']:
        labels = assignment.loc[group.index, name + '_cluster'].to_numpy()
        for cluster in np.unique(labels):
            member = labels == cluster
            for feature in FEATURES:
                for category in range(3 if feature in WORRY else 2):
                    proportion = np.average(group.loc[member, feature].eq(category), weights=mass[member])
                    profile_rows.append({'split': split, 'candidate': name, 'cluster': int(cluster),
                                         'feature': feature, 'category': category, 'proportion': proportion})
profiles = pd.DataFrame(profile_rows)
profiles.to_csv(OUT / 'week05_recomputed_original_profiles.csv', index=False)
for split, filename in [('train', FROZEN / 'development_original_category_profiles.csv'),
                         ('heldout_country', HELDOUT / 'heldout_original_category_profiles.csv')]:
    original = pd.read_csv(filename)
    original = original.loc[original.scope.eq('global')]
    compared = profiles.loc[profiles.split.eq(split)].merge(original, on=['candidate', 'cluster', 'feature', 'category'],
                                                           suffixes=('_new', '_saved'), validate='one_to_one')
    assert len(compared) == 552 and np.allclose(compared.proportion_new, compared.proportion_saved, atol=1e-13)
print('All 1,104 global original-category profile proportions reproduced.')


### A.6. Compare countries without changing respondent profiles

    Two different distances answer different questions. Hellinger distance
    compares country mixtures over the same frozen profiles: $H(p,q)=
    \sqrt{\tfrac12\sum_j(\sqrt{p_j}-\sqrt{q_j})^2}$. Similar mixtures need
    not mean similar original responses; broad profiles compress information.

    Direct-response distance compares each original feature's category
    distribution: $T_f(c,d)=\tfrac12\sum_r|p_{cfr}-p_{dfr}|$.
    Average these over six worry, seven personal-harm and seven known-person-harm
    features to obtain W, S and K. Overall distance is
    $D_\alpha=\alpha W+(1-\alpha)(S+K)/2$, with alpha .50 for B and .75 for E.
    These are marginal distributions, not complete joint response distributions.
    No extra 0/1/3 category spacing enters total variation; the B/E allocation
    sets the component weights only.

    The display divides B/E distances by their pooled distinct-pair median over
    all 142 countries (.1550977481). Components use their own all-country medians.
    The reference remains fixed across countries and development/all-country
    views. Ratios are uncapped; only the colour legend ends at “2 or more”.
    This changes visibility, not raw distances or nearest-country order.
    All-country distance/income exploration is post-validation descriptive work,
    not another untouched validation experiment. Country pairs are dependent.


In [ ]:
DIST = ART / 'week05_country_response_distances_v1/e1ecf01551bcf2da'
probabilities = pd.read_csv(DIST / 'country_response_probabilities.csv')
countries = sorted(probabilities.COUNTRY_ISO3.unique())
feature_distances = []
country_mass = data.WGT / data.groupby('COUNTRY_ISO3').WGT.transform('sum')
for feature in FEATURES:
    table = probabilities.loc[probabilities.feature.eq(feature)].pivot(
        index='COUNTRY_ISO3', columns='category', values='proportion').loc[countries]
    # Reconstruct probabilities from respondents, not just from the saved matrix.
    current = data.assign(mass=country_mass).groupby(['COUNTRY_ISO3', feature]).mass.sum().unstack(fill_value=0)
    current = current.reindex(index=countries, columns=table.columns, fill_value=0)
    assert np.allclose(current, table, atol=1e-14)
    p = current.to_numpy()
    assert np.allclose(p.sum(axis=1), 1) and (p >= 0).all()
    feature_distances.append(np.abs(p[:, None, :] - p[None, :, :]).sum(axis=2) / 2)
distances = {'worry': np.mean(feature_distances[:6], axis=0),
             'self': np.mean(feature_distances[6:13], axis=0),
             'known': np.mean(feature_distances[13:], axis=0)}
for name, alpha in [('B', .5), ('E', .75)]:
    distances[name] = alpha * distances['worry'] + (1 - alpha) * (distances['self'] + distances['known']) / 2
for name, matrix in distances.items():
    original = pd.read_csv(DIST / f'{name}_all_countries_post_validation_matrix.csv', index_col=0).loc[countries, countries]
    assert np.allclose(matrix, original, atol=1e-14)
pairs = np.triu_indices(len(countries), 1)
median = np.median(np.concatenate([distances[name][pairs] for name in ['B', 'E']]))
assert np.isclose(median, .1550977480973838, atol=1e-14)
print('Pooled B/E distance median:', median)
display(pd.read_csv(DIST / 'response_income_summary.csv').round(3))


In [ ]:
# Also reconstruct the older profile-mixture Hellinger distance for both models.
mixture_source = ART / 'week05_country_similarity_and_profile_synthesis_v1/d694096841b06e54'
vectors = pd.read_csv(mixture_source / 'country_profile_vectors.csv')
for name in ['B', 'E']:
    table = assignment.assign(mass=country_mass).groupby(['COUNTRY_ISO3', name + '_cluster']).mass.sum().unstack(fill_value=0)
    table = table.reindex(index=countries, columns=range(bundle['models'][name]['spec']['global_k']), fill_value=0)
    saved_vectors = vectors.loc[vectors.candidate.eq(name)].pivot(index='COUNTRY_ISO3', columns='cluster', values='profile_share')
    assert np.allclose(table, saved_vectors.loc[table.index, table.columns], atol=1e-14)
    hellinger = cdist(np.sqrt(table), np.sqrt(table)) / np.sqrt(2)
    assert np.allclose(np.diag(hellinger), 0)
    print(name, 'profile-mixture Hellinger range:', hellinger.min(), hellinger.max())


In [ ]:
# Illustration: five nearest countries to the USA.
for name in ['B', 'E']:
    i = countries.index('USA')
    nearest = sorted((j for j in range(len(countries)) if j != i), key=lambda j: (distances[name][i, j], countries[j]))[:5]
    display(pd.DataFrame({'model': name, 'country': [countries[j] for j in nearest],
                          'raw_distance': [distances[name][i, j] for j in nearest],
                          'median_relative_distance': [distances[name][i, j] / median for j in nearest]}))


### A.7. Final handoff and limits

    Use the existing ID-linked assignment table, not row position or freshly
    renumbered labels. It retains excluded respondents, reasons for exclusion,
    split, stable B/E IDs, profile names, distances and margins. Week 6 reads this
    same frozen handoff. This rewrite does not modify it.

    The global profiles transfer reasonably in their original response patterns;
    this does not establish universal regional optima. Regional optimized models
    often prefer coarser structures. Missingness selection, limited country
    resampling in small regions and unequal local fit remain limitations. Keep
    the shared global definitions for cross-country comparison and treat local
    differences as scientific results, not a reason to retune on held-out data.

    Existing interactive results (unchanged):
    [profile explorer](../github_publication/docs/index.html),
    [country response map](../github_publication/docs/countries.html).
    The archived notebook contains the full profile synthesis, country-tree,
    uncertainty and income analyses, including their mathematical explanations.


## Check of this simplification — 16 October 2026

    All five notebooks run from top to bottom with expensive refits disabled.
    Fresh checks reproduced 19 K-means reference partitions exactly (ARI 1.000),
    as well as two GMM references and one DBSCAN reference. Ward and average
    linkage reproduced their saved labels using the original bootstrap/tree.
    The shallow B/E trees and single-feature rules matched to floating-point
    precision. All 274,920 frozen B/E assignments matched exactly. Original-unit
    profile proportions and country-response distances were also checked.

    The original full stability searches were **not** rerun. Their saved evidence
    remains unchanged; standard scikit-learn ARI was checked against 324 cached
    pair comparisons covering every encoding, allocation, k and test.
    See the [verification report](../course_artifacts/objective1_simplified_checks/verification_report.json).


## Readability revision checked

All displayed tables, plots and printed results match the preceding version exactly. The revised GMM/DBSCAN functions also reproduced the tested reference labels exactly (ARI 1.000). No full stability search was repeated. See the [readability check](../course_artifacts/objective1_simplified_checks/readability_pass_report.json).